In [209]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)


In [210]:
csv_path="fraud_dataset.csv"
df=pd.read_csv(csv_path)


In [211]:
#Replace whitespaces, mutilple hypens etc with one single whitespace.
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)


In [212]:
#Checking number of rows and columns.
print(f"{df.shape}")


(5000, 13)


In [213]:
df.dtypes


transaction_id          str
customer_id             str
amount              float64
customer_age        float64
past_fraud_count      int64
avg_spend           float64
is_international      int64
device_type             str
customer_region         str
category                str
merchant                str
transaction_time        str
is_fraud              int64
dtype: object

In [214]:
#Checking overall columns data.
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    5000 non-null   str    
 1   customer_id       5000 non-null   str    
 2   amount            5000 non-null   float64
 3   customer_age      4500 non-null   float64
 4   past_fraud_count  5000 non-null   int64  
 5   avg_spend         5000 non-null   float64
 6   is_international  5000 non-null   int64  
 7   device_type       5000 non-null   str    
 8   customer_region   5000 non-null   str    
 9   category          4500 non-null   str    
 10  merchant          4500 non-null   str    
 11  transaction_time  5000 non-null   str    
 12  is_fraud          5000 non-null   int64  
dtypes: float64(3), int64(3), str(7)
memory usage: 507.9 KB


In [215]:
#Checking how the data is distributed in all numerical columns
df.describe()


,amount,customer_age,past_fraud_count,avg_spend,is_international,is_fraud
count,5000.000000,4500.000000,5000.000000,5000.000000,5000.00000,5000.000000
mean,191.972524,43.141778,0.200600,100.872632,0.15640,0.010800
std,960.205272,14.919474,0.442718,50.436836,0.36327,0.103371
min,0.000000,18.000000,0.000000,-81.760000,0.00000,0.000000
25%,28.717500,30.000000,0.000000,67.147500,0.00000,0.000000
50%,70.865000,43.000000,0.000000,100.900000,0.00000,0.000000
75%,143.140000,56.000000,0.000000,134.575000,0.00000,0.000000
max,27166.000000,69.000000,3.000000,323.950000,1.00000,1.000000


In [216]:
#Checking the top 5 rows
df.head()


,transaction_id,customer_id,amount,customer_age,past_fraud_count,avg_spend,is_international,device_type,customer_region,category,merchant,transaction_time,is_fraud
0,T1000,C291,46.93,68.0,0,124.55,1,mobile,North,travel,Walmart,01/01/2024 08:10,0
1,T1001,C280,301.01,57.0,1,116.49,0,pos,North,travel,NaN,2024-01-01 07:50:12,0
2,T1002,C321,131.67,NaN,1,96.91,0,web,North,grocery,Amazon,2024-01-01 18:09:06,0
3,T1003,C450,91.29,49.0,0,108.17,0,pos,West,electronics,NaN,2024/01/01 03:52,0
4,T1004,C111,16.96,65.0,0,70.14,0,pos,South,grocery,Amazon,Jan 01 2024 09:08,0


In [217]:
#Checking sum of null values in each column
df.isna().sum()


transaction_id        0
customer_id           0
amount                0
customer_age        500
past_fraud_count      0
avg_spend             0
is_international      0
device_type           0
customer_region       0
category            500
merchant            500
transaction_time      0
is_fraud              0
dtype: int64

In [218]:
#Check percentage of missing values in each column.
missing_perc=((df.isnull().sum() / len(df)) * 100)
missing_perc[missing_perc>0]


customer_age    10.0
category        10.0
merchant        10.0
dtype: float64

In [219]:
df["transaction_time"].info()
print("\n------------------------------------------------------------------------------------------\n")

#Fix the datetime format.
df['transaction_time'] = pd.to_datetime(df['transaction_time'], format='mixed', dayfirst=True , errors='coerce')
df["transaction_time"].info()


<class 'pandas.Series'>
RangeIndex: 5000 entries, 0 to 4999
Series name: transaction_time
Non-Null Count  Dtype
--------------  -----
5000 non-null   str  
dtypes: str(1)
memory usage: 39.2 KB

------------------------------------------------------------------------------------------

<class 'pandas.Series'>
RangeIndex: 5000 entries, 0 to 4999
Series name: transaction_time
Non-Null Count  Dtype         
--------------  -----         
5000 non-null   datetime64[us]
dtypes: datetime64[us](1)
memory usage: 39.2 KB


In [220]:
#handling missing values.
df['customer_age']=df['customer_age'].fillna(df['customer_age'].median())

#Replaced missing values in category and merchant columns with "Unknown" str 
#cause there were 500 missing values in both columns 
#with the mode in both been closest to the second largest value 
#so giving it the value "Unknown" seems better than some very specific values and creating bias.
df['category']=df['category'].fillna("Unknown")
df['merchant']=df['merchant'].fillna("Unknown")


In [221]:
#Checking if there are any missing values left even after replacing.
df.isna().sum()


transaction_id      0
customer_id         0
amount              0
customer_age        0
past_fraud_count    0
avg_spend           0
is_international    0
device_type         0
customer_region     0
category            0
merchant            0
transaction_time    0
is_fraud            0
dtype: int64

In [222]:
outliers_columns=df.select_dtypes(include='number').drop(columns=['transaction_id','customer_id','is_fraud','is_international'], errors='ignore').columns
outliers_columns


Index(['amount', 'customer_age', 'past_fraud_count', 'avg_spend'], dtype='str')

In [ ]:
total_outliers=0
for column in outliers_columns:
#Dectecting outliers using basic Interquartile Range Method in the amount column.
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df[
        (df[column] < lower) |
        (df[column] > upper)
    ]
    total_outliers+=len(outliers)
    outliers.to_csv(f"{column}_outliers.csv")


In [224]:
print(total_outliers)


1265


In [225]:
categorical_cols=['device_type','customer_region','category','merchant']


In [226]:
for column in categorical_cols:
    df[column] = df[column].str.strip().str.lower()


In [227]:
for column in categorical_cols:
    print(f"\nColumn: {column}")
    print(f"Number of unique values: {df[column].nunique()}")
    print("Unique values:")
    print(df[column].unique())
    print(f"\nColumn: {column}")
    print(df[column].value_counts())



Column: device_type
Number of unique values: 3
Unique values:
<StringArray>
['mobile', 'pos', 'web']
Length: 3, dtype: str

Column: device_type
device_type
web       1684
pos       1674
mobile    1642
Name: count, dtype: int64

Column: customer_region
Number of unique values: 4
Unique values:
<StringArray>
['north', 'west', 'south', 'east']
Length: 4, dtype: str

Column: customer_region
customer_region
south    1277
east     1263
west     1248
north    1212
Name: count, dtype: int64

Column: category
Number of unique values: 6
Unique values:
<StringArray>
['travel', 'grocery', 'electronics', 'shopping', 'food', 'unknown']
Length: 6, dtype: str

Column: category
category
electronics    979
grocery        915
travel         878
food           867
shopping       861
unknown        500
Name: count, dtype: int64

Column: merchant
Number of unique values: 6
Unique values:
<StringArray>
['walmart', 'unknown', 'amazon', 'target', 'starbucks', 'bestbuy']
Length: 6, dtype: str

Column: merchant

In [228]:
ohe= OneHotEncoder(
    sparse_output=False,
    handle_unknown='ignore'
)


In [ ]:
encoded_data = ohe.fit_transform(df[categorical_cols])
